(course-core-system-modification)=

# Module 10: Modifying Molecular Attributes

**Welcome back, Apprentice Master.** In [Module 8: Extracting Molecular Attributes](../Common_Core/08_Extracting_Molecular_Attributes.ipynb), you learned how to retrieve system attributes using `msm.get()`. Now we explore the direct counterpart operation: **Modifying Molecular Attributes** using **`msm.set()`**.

Molecular modelling and trajectory analysis frequently require editing a system's state: adjusting 3D atomic coordinates, standardizing chain identifiers, or renaming chemical groups.

(course-core-system-modification-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Understand form modifiability and why editing in-memory systems requires a modifiable form (`molsysmt.MolSys`).
- Modify atomic coordinates using `msm.set(system, element='atom', selection=..., coordinates=...)`.
- Update group, chain, and entity identifiers (`chain_id`, `group_name`, `entity_name`).
- Duplicate molecular systems safely using `msm.copy()` before applying in-place modifications.
- Verify structural modifications using `msm.get()` and `msm.info()`.
```

### 1. Modifiable Forms

Let's begin by importing MolSysMT and loading our T4 Lysozyme demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme file
lysozyme = systems['T4 lysozyme L99A']['181l.bcif.gz']

:::{important}
**Crucial Concept: Modifiable Forms**  
Not all molecular data forms are modifiable. Static disk files (such as a read-only PDB or BCIF file) cannot be edited directly in place. To modify attributes or coordinates, you must first convert the system into an in-memory modifiable form, such as **`molsysmt.MolSys`**.
:::
After recalculating selected frames on the original system axes and atom scope, use `current.replace_structures(fresh)` and explicitly attach its returned analysis under the same name. It replaces only incoming evaluated frames (including empty results), preserves other frames, and checks that method, parameters, producer versions, units and provenance match. Directly assigning `fresh` under the name replaces the entire analysis.


In [2]:
# Convert system into an in-memory modifiable native object
molsys = msm.convert(lysozyme, to_form='molsysmt.MolSys')

:::{hint}
:class: dropdown
**msm.set()**: Form-agnostic attribute mutator engine. Updates coordinates, topology identifiers, or physical properties on modifiable molecular systems. See API doc: {func}`molsysmt.basic.set`.
:::

### 2. Modifying Coordinates

You can update the 3D position of any atom or group using `msm.set()`. As established in Module 9, input quantities are quantity-agnostic and accept explicit unit strings:

For a native MolSys with named interactions, coordinate and box edits through `msm.set()` remove observations and evaluated coverage only in the selected structures. Previously evaluated-empty structures also become unevaluated. Untouched structures and old query snapshots survive; recalculate before attaching new observations. Compatible partial recalculations can use different execution block sizes: `Interactions.replace_structures()` preserves each run in `execution_records`, while requiring matching scientific parameters, producer versions and atom scope. Empty selections and identifier/time edits preserve analyses. Attached `structures` coordinate/box setters notify the owner; raw/private writes and whole-domain replacement still need explicit invalidation.
Invalidation shares read-only observation columns rather than copying all surviving rows. Moving an atom that had no previous interaction still invalidates its frame because new interactions can appear. The same automatic policy applies to the attached `molsys.structures.set_coordinates()` / `set_box()` and full coordinate/box property assignments, including calls through a domain alias. Standalone domain copies detach, while complete system copies bind their copied domains to the new system. Direct native setters validate before publishing geometry and invalidation. Raw/private writes, whole-domain replacement and direct domain append/add remain outside this boundary; use coordinated `MolSys` operations when changing axes. No detector runs automatically. H5MSM saving writes active interaction blocks without packing all edited observations; typed dictionary conversion, pickle and remapping may still materialize them. After repeated edits, you can explicitly attach `analysis.compact()` to reclaim retired observation blocks once old analyses and query views are released. Compaction preserves indices, coverage and provenance, but needs memory for its new columns and retains unused relation definitions. Read the [coordinate edit policy](../../user/tools/interactions/result.md#changing-coordinates) for coverage checks, explicit invalidation and recalculation rules.


Chemical-state atom and scientific bond edits with `msm.set(molsys, ...)` or on its attached topology
invalidate every covered structure in the named analyses. Replacing
`molsys.chemical_states` does the same. Changing
`structure_chemical_state_index` invalidates only the selected frames.
Bond IDs and empty selections preserve results. No detector runs
automatically. Controlled chemical-state assignments on the attached topology, native bond addition/removal and bond-table replacement notify the owner too. Invalid values and invalidation allocation failures preserve chemical values; bond tools can fail during subsequent component rebuilding with bonds already published and analyses invalidated. Appending a chemical state through `molsys.chemical_states.append_state()` invalidates all named analyses without copying existing states or coordinates. Explicit structure-to-state assignments remain unchanged; with multiple states, missing assignments resolve to unknown. Appending preparation history preserves analyses because it only archives evidence. Other private collection changes, whole-topology replacement, mechanics changes and raw data writes need explicit owner invalidation before new calculations.

After recalculating selected frames on the original system axes and atom scope, use `current.replace_structures(fresh)` and explicitly attach its returned analysis under the same name. It replaces only incoming evaluated frames (including empty results), preserves other frames, and checks that method, parameters, producer versions, units and provenance match. Directly assigning `fresh` under the name replaces the entire analysis.


In [3]:
# Move atom index 0 to the spatial origin
msm.set(molsys, element='atom', selection=0, coordinates='[0.0, 0.0, 0.0] nm')

# Verify updated position using msm.get()
updated_pos = msm.get(molsys, element='atom', selection=0, coordinates=True)
print(f"Updated position for atom 0: {updated_pos}")

Updated position for atom 0: [[[0.0 0.0 0.0]]] nanometer


### 3. Modifying Attributes

Structural files retrieved from public databases often contain non-standard or ambiguous chain or group identifiers. You can standardize these properties programmatically:

In [4]:
# Rename Chain 'A' to 'X'
msm.set(molsys, element='chain', selection='chain_id == "A"', chain_id='X')

# Verify updated chain topology with msm.info()
msm.info(molsys, element='chain')

index,id,name,n atoms,n groups,n components,molecule index,molecule type,entity index,entity name
0,X,A,1289,162,1,[0],['protein'],[0],['T4 LYSOZYME']
1,B,A,1,1,1,[1],['ion'],[1],['CHLORIDE ION']
2,C,A,1,1,1,[2],['ion'],[1],['CHLORIDE ION']
3,D,A,8,1,1,[3],['small molecule'],[2],['2-HYDROXYETHYL DISULFIDE']
4,E,A,6,1,1,[4],['small molecule'],[3],['BENZENE']
5,F,A,136,136,136,"[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140]","['water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water']",[4],['water']


### 4. Duplicating Systems Before Modification

In Python, variable assignment (`sys_b = sys_a`) creates a shared reference alias rather than a new object. Modifying `sys_b` will alter `sys_a`. To preserve an unmodified original state before calling `msm.set()`, create a deep independent copy with **`msm.copy()`**:

File paths have a different boundary: `msm.copy(source_filename)` creates a
private process-owned temporary, keeping its bytes and version. It remains
available through the session; normal-exit cleanup is best effort. An explicit
`output_filename` instead creates a persistent caller-owned copy. Composite
sources receive separate copies of their members. A copy does not add write
support to a read-only adapter: use `molsysmt.MolSys` when needed for editing.
See {ref}`Cookbook_Copying_Disk_Systems`.


In [5]:
# Create an independent copy of the system
molsys_backup = msm.copy(molsys)

# Modify the backup copy independently
msm.set(molsys_backup, element='chain', selection='chain_id == "X"', chain_id='Z')

# Verify that original molsys retains chain_id 'X'
orig_chain = msm.get(molsys, element='chain', selection=0, chain_id=True)
copy_chain = msm.get(molsys_backup, element='chain', selection=0, chain_id=True)
print(f"Original chain ID: {orig_chain} | Backup chain ID: {copy_chain}")

Original chain ID: ['X'] | Backup chain ID: ['Z']


:::{hint}
:class: dropdown
**msm.copy()**: Creates an independent deep copy of a molecular system object in memory. See API doc: {func}`molsysmt.basic.copy`.
:::

--- 

### 🏆 Challenge 10: The Molecular Editor

1. Load the **T4 Lysozyme** system (`systems['T4 lysozyme L99A']['181l.bcif.gz']`) and convert it to a native `MolSys` object.
2. Make a deep copy using `msm.copy()`.
3. Use `msm.set()` to rename the water chain (`selection='molecule_type == "water"'`) in the copy to `'W'`.
4. Set the coordinates of atom index 10 to `'[1.0, 1.0, 1.0] nm'` in the copy.
5. Verify your modifications using `msm.get()` and check that the original object remains unchanged.

System modification is a key step in preparing systems for simulation or building virtual complexes. In [Module 11: Iterating Systems](../Common_Core/11_Iterating_Systems.ipynb), we will learn how to stream heavy systems iteratively.

```{key-takeaway}
The `msm.set()` function is MolSysMT's universal attribute mutator. Always ensure your system is converted to a modifiable form (such as `molsysmt.MolSys`), and use `msm.copy()` when you need to preserve an unmodified backup before applying in-place modifications.
```

(course-core-system-modification-mechanical-labels)=
### Replacing mechanical labels

Mechanical atom labels are parameter assignments, separate from element symbols.
For a native MolecularMechanics object, `msm.set(..., element='atom',
atom_ff_type=...)` accepts a one-dimensional string vector. Subset values follow
the requested index order and require an existing atom axis. Invalid lengths
fail before mutation. Manual labels are a caller declaration, and replacement
clears their named typing provenance; it does not certify a chemical scheme.
See {ref}`Tutorial_Assign_AutoDock_Types` for calculated assignments and export.


### String atom labels

An atom ID is a label, while its index is a position. Native ID edits accept integers and strings and store strings, preserving lexical zeros and unselected atoms. The independent example below uses public setters and confirms that source indices are unchanged.


In [6]:
import molsysmt as msm
molsys = msm.convert(msm.systems['T4 lysozyme L99A']['181l.pdb'], to_form='molsysmt.MolSys')
molsys = msm.extract(molsys, selection=[0, 1])
msm.set(molsys, selection=[0], atom_id=101)
msm.set(molsys, selection=[1], atom_id='00101')
labels = msm.get(molsys, element='atom', atom_id=True)
print(labels)
assert labels == ['101', '00101']
assert msm.get(molsys, element='atom', atom_index=True) == [0, 1]


['101', '00101']


(course-core-system-modification-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.set` — Universal attribute mutation engine.
- {func}`molsysmt.basic.copy` — Deep system copying tool.

**Related Course Modules & Guides:**
- Previous Module: [Module 9: Physical Unit Safety](../Common_Core/09_Unit_Safety.ipynb)
- Next Module: [Module 11: Iterating Systems](../Common_Core/11_Iterating_Systems.ipynb)
- User Guide: {ref}`user-foundations-index`
:::